In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

C:\Users\hp\AppData\Local\Temp\ipykernel_8492\4123191527.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
# inbuilt tools
import wikipedia
wikipedia.set_user_agent("GenerativeAINotesBot/1.0 (contact@example.com)")

import arxiv
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

print(wiki.name)

wikipedia


In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=300)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

print(arxiv.name)

arxiv


In [4]:
# custom tools -> retriever_tool [RAG TOOL]
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\Desktop\AI\Generative-ai-notes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
loader = WebBaseLoader("https://www.langchain.com/blog/introducing-langserve")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6666.25it/s]


In [6]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="Langserve-search",
    description="Search any information about Langserve."
)

print(retriever_tool.name)

Langserve-search


In [7]:
tools = [wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Desktop\\AI\\Generative-ai-notes\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=300)),
 StructuredTool(name='Langserve-search', description='Search any information about Langserve.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001CB12CDA3E0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001CB146FA8E0>)]

In [8]:
# Run all this tools with Agents and LLM Models

from langchain_ollama import ChatOllama

import os
from dotenv import load_dotenv
load_dotenv()

model = ChatOllama(model="gemma4:31b-cloud")

In [9]:
# PROMPT
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [10]:
prompt.get_prompts()

[ChatPromptTemplate(input_variables=['input'], optional_variables=['agent_scratchpad'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessa

In [11]:
from langchain_classic.agents import create_tool_calling_agent
agent = create_tool_calling_agent(
    model,
    tools,
    prompt=prompt
)

In [12]:
# Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True, handle_parsing_errors=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['input'], optional_variables=['agent_scratchpad'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')

In [84]:
agent_executor.invoke({"input":"What is machine learning?"})



> Entering new AgentExecutor chain...
**Machine Learning (ML)** is a branch of artificial intelligence (AI) focused on building systems that can "learn" from data. 

In traditional computer programming, a human writes a specific set of instructions (rules) for the computer to follow to reach a goal. In machine learning, instead of writing those rules by hand, you feed the computer large amounts of data and use an **algorithm** to let the computer discover the patterns and rules itself.

Here is a breakdown of how it works, the different types, and where you see it in real life.

---

### 1. How does it work?
The core process of machine learning involves three main components:
*   **Data:** The "textbook" the machine studies. This could be images, numbers, text, or sensor readings.
*   **Model:** A mathematical representation (like a complex equation) that tries to describe the relationship between the data.
*   **Training:** The process of showing the model data and correcting it whe

{'input': 'What is machine learning?',
 'output': '**Machine Learning (ML)** is a branch of artificial intelligence (AI) focused on building systems that can "learn" from data. \n\nIn traditional computer programming, a human writes a specific set of instructions (rules) for the computer to follow to reach a goal. In machine learning, instead of writing those rules by hand, you feed the computer large amounts of data and use an **algorithm** to let the computer discover the patterns and rules itself.\n\nHere is a breakdown of how it works, the different types, and where you see it in real life.\n\n---\n\n### 1. How does it work?\nThe core process of machine learning involves three main components:\n*   **Data:** The "textbook" the machine studies. This could be images, numbers, text, or sensor readings.\n*   **Model:** A mathematical representation (like a complex equation) that tries to describe the relationship between the data.\n*   **Training:** The process of showing the model dat

In [85]:
agent_executor.invoke({"input":"Why Langserve exists?"})



> Entering new AgentExecutor chain...

Invoking: `Langserve-search` with `{'query': 'what is langserve and why does it exist'}`


the past week, and this has help battle test it.The road to LangServeThe journey to build LangServe really started a few months ago when we launched LCEL and the Runnable protocol. This was designed from day 1 to support putting prototypes in production, with no code changes, from the simplest “prompt + LLM” chain to the most complex chains (we’ve seen folks successfully running in production LCEL chains with 100s of steps). To highlight a few:first-class support for streaming: when you build your chains with LCEL you get the best possible time-to-first-token (time elapsed until the first chunk of output comes out). For some chains this means eg. we stream tokens straight from an LLM to a streaming output parser, and you get back parsed, incremental chunks of output at the same rate as the LLM provider outputs the raw tokens. We’re constantly improving str

{'input': 'Why Langserve exists?',
 'output': 'LangServe exists to bridge the gap between **prototyping** an LLM application and **deploying** it as a production-ready API. \n\nEssentially, it is designed to turn your LangChain chains (built using the LangChain Expression Language, or LCEL) into a REST API with minimal effort. Here is the detailed breakdown of why it exists:\n\n### 1. Rapid Transition from Prototype to Production\nIn a typical development cycle, you might build a complex chain in a Jupyter notebook or a script. Traditionally, moving that logic into a production server would require manually writing FastAPI or Flask wrappers, defining input/output schemas, and handling serialization. LangServe automates this process, allowing you to go from **"Idea + LCEL" $\\rightarrow$ "Production-ready API"** almost instantly.\n\n### 2. Standardizing the "Runnable" Protocol\nLangServe is built on top of the **Runnable protocol** (the foundation of LCEL). Because all LCEL chains follo

In [87]:
agent_executor.invoke({"input":"Explain Gradient Descent."})



> Entering new AgentExecutor chain...
**Gradient Descent** is an iterative optimization algorithm used to minimize a function. In the context of Machine Learning, it is the primary tool used to train models by minimizing the **Cost Function** (the error between the model's predictions and the actual target values).

Here is a comprehensive explanation broken down into simple concepts.

---

### 1. The Intuition: The Mountain Analogy
Imagine you are standing at the top of a mountain (the high error state) and you want to get down to the lowest valley (the minimum error state). However, there is a thick fog, and you cannot see the valley.

To get down, you would:
1. Feel the slope of the ground under your feet.
2. Take a small step in the direction where the slope goes **downward**.
3. Repeat this process until the ground feels flat under your feet.

In this analogy:
*   **The Mountain** = The Cost Function.
*   **The Steepness/Slope** = The Gradient.
*   **The Step Size** = The Learni

{'input': 'Explain Gradient Descent.',
 'output': '**Gradient Descent** is an iterative optimization algorithm used to minimize a function. In the context of Machine Learning, it is the primary tool used to train models by minimizing the **Cost Function** (the error between the model\'s predictions and the actual target values).\n\nHere is a comprehensive explanation broken down into simple concepts.\n\n---\n\n### 1. The Intuition: The Mountain Analogy\nImagine you are standing at the top of a mountain (the high error state) and you want to get down to the lowest valley (the minimum error state). However, there is a thick fog, and you cannot see the valley.\n\nTo get down, you would:\n1. Feel the slope of the ground under your feet.\n2. Take a small step in the direction where the slope goes **downward**.\n3. Repeat this process until the ground feels flat under your feet.\n\nIn this analogy:\n*   **The Mountain** = The Cost Function.\n*   **The Steepness/Slope** = The Gradient.\n*   *

In [13]:
agent_executor.invoke({"input":"Tell me about arXiv:1706.03762"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks arXiv:1706.03762 is the identifier for one of the most influential papers in the history of modern artificial intelligence: **"Attention Is All You Need."**

Published in 2017 by researchers from Google Brain and Google Research, this paper introduced the **Transformer architecture**, which fundamentally changed how machines process natural language and paved the way for the current era of Large Language Models (LLMs) like GPT-4, Claude, and Gemini.

### Key Contributions and Concepts:

**1. The Transformer Architecture**
Before this paper, the state-of-the-art for sequence tasks (like translation) re

{'input': 'Tell me about arXiv:1706.03762',
 'output': 'arXiv:1706.03762 is the identifier for one of the most influential papers in the history of modern artificial intelligence: **"Attention Is All You Need."**\n\nPublished in 2017 by researchers from Google Brain and Google Research, this paper introduced the **Transformer architecture**, which fundamentally changed how machines process natural language and paved the way for the current era of Large Language Models (LLMs) like GPT-4, Claude, and Gemini.\n\n### Key Contributions and Concepts:\n\n**1. The Transformer Architecture**\nBefore this paper, the state-of-the-art for sequence tasks (like translation) relied on Recurrent Neural Networks (RNNs) or Convolutional Neural Networks (CNNs). RNNs processed data sequentially (word by word), which was slow and struggled with "forgetting" the beginning of a long sentence. The Transformer abandoned recurrence entirely in favor of a structure based solely on **attention mechanisms**.\n\n**